In [1]:
!python3 --version

Python 3.8.10


#### CSV PARSING

In [3]:
import csv
import pandas as pd

with open("reviews.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    rows = list(reader)

header = rows[0]
data_rows = rows[1:]

csv_valid_rows = []
csv_malformed_rows = []

for row_number, row in enumerate(data_rows, start=2):
    if len(row) == 14:
        csv_valid_rows.append(row)
    else:
        csv_malformed_rows.append({
            "csv_row": row_number,
            "field_count": len(row),
            "row": row
        })

df = pd.DataFrame(csv_valid_rows, columns=header)

print("Header fields:", len(header))
print("Total records:", len(data_rows))
print("CSV-valid-rows:", len(csv_valid_rows))
print("CSV-malformed-rows:", len(csv_malformed_rows))
print("DataFrame shape:", df.shape)

Header fields: 14
Total records: 10548
CSV-valid-rows: 10521
CSV-malformed-rows: 27
DataFrame shape: (10521, 14)


##### There are 10,548 CSV records and 14 headers.
##### 27 CSV-rows are malformed.
##### Therefore, those 27 CSV-rows are malformed at the CSV-parsing level and must be discarded.

##### CHECKING FOR MALFORMED ROWS BEFORE CSV PARSING

In [ ]:
with open("reviews.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    
    field_counts = []
    
    for row in reader:
        field_counts.append(len(row))

print("Number of data records:", len(field_counts) - 1)
print("Unique field counts:", set(field_counts))

In [ ]:
malformed_csv_rows = []

with open("reviews.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)

    for row_number, row in enumerate(reader, start=1):
        if len(row) != 14:
            malformed_csv_rows.append({
                "csv_row": row_number,
                "field_count": len(row),
                "row": row
            })

len(malformed_csv_rows)

In [ ]:
for item in malformed_csv_rows[:5]:
    print("CSV record:", item["csv_row"])
    print("Field count:", item["field_count"])
    print("Fields:", item["row"])
    print("-" * 80)

In [ ]:
from collections import Counter

Counter(item["field_count"] for item in malformed_csv_rows)

##### This means all malformed 27 records have only 9 fields.

In [ ]:
for item in malformed_csv_rows:
    print("CSV record:", item["csv_row"])
    print("Field count:", item["field_count"])
    print(item["row"])
    print()

In [ ]:
df = pd.read_csv('reviews.csv')

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

#### SCHEMA LEVEL VALIDATION

##### Validate submission_id and reviewer_id

In [ ]:
valid_submission_id = (
    df["submission_id"].notna()
    & df["submission_id"].astype(str).str.strip().ne("")
)

valid_reviewer_id = (
    df["reviewer_id"].notna()
    & df["reviewer_id"].astype(str).str.strip().ne("")
)

##### Validate track and recommendation

In [ ]:
allowed_tracks = {
    "MainConference",
    "Findings",
    "Workshop",
    "Demo"
}

allowed_recommendations = {
    "Strong Accept",
    "Accept",
    "Borderline",
    "Reject",
    "Strong Reject"
}

valid_track = df['track'].isin(allowed_tracks)
valid_recommendation = df["recommendation"].isin(
    allowed_recommendations
)

In [ ]:
(~valid_track).sum()

In [ ]:
df.loc[~valid_track, ["submission_id", "track"]]

In [ ]:
(~valid_recommendation).sum()

In [ ]:
#Inspect malformity

df.loc[~valid_recommendation, ["submission_id", "recommendation"]].head(2)

In [ ]:
assignment_dt = pd.to_datetime(
    df["assignment_timestamp"],
    errors="coerce",
    utc=True
)

review_dt = pd.to_datetime(
    df["review_timestamp"],
    errors="coerce",
    utc=True
)

In [ ]:
valid_assignment_timestamp = assignment_dt.notna()
valid_review_timestamp = review_dt.notna()

In [ ]:
(~valid_assignment_timestamp).sum()

In [ ]:
(~valid_review_timestamp).sum()

In [ ]:
df.loc[~valid_assignment_timestamp, ["submission_id", "assignment_timestamp"]].head(2)

In [ ]:
df.loc[~valid_review_timestamp, ["submission_id", "review_timestamp"]].head(2)

##### Validate soundness, excitement and confidence

In [ ]:
def valid_integer_score(label, lower=1, upper=5):
    parsed = pd.to_numeric(label, errors="coerce")
    
    return (
        parsed.notna()
        & (parsed % 1 == 0)
        & parsed.between(lower, upper)
    )

In [ ]:
valid_soundness = valid_integer_score(df["soundness"])
valid_excitement = valid_integer_score(df["excitement"])
valid_confidence = valid_integer_score(df["confidence"])

In [ ]:
(~valid_soundness).sum()

In [ ]:
(~valid_excitement).sum()

In [ ]:
(~valid_confidence).sum()

In [ ]:
df['soundness'].value_counts()

In [ ]:
df.loc[~valid_soundness, ["submission_id", "soundness"]].head(2)

In [ ]:
df.loc[~valid_excitement, ["submission_id", "excitement"]]

In [ ]:
df.loc[~valid_confidence, ["submission_id", "confidence"]].head(2)

##### Validate overall_score and metareview_score

In [ ]:
def valid_float_score(label, lower=1.0, upper=10.0):
    parsed = pd.to_numeric(label, errors="coerce")
    
    return (
        parsed.notna()
        & parsed.between(lower, upper)
    )

In [ ]:
valid_overall_score = valid_float_score(
    df["overall_score"]
)

valid_metareview_score = valid_float_score(
    df["metareview_score"]
)

In [ ]:
(~valid_overall_score).sum()

In [ ]:
(~valid_metareview_score).sum()

In [ ]:
df.loc[~valid_overall_score, ["submission_id", "overall_score"]]

In [ ]:
df.loc[~valid_metareview_score, ["submission_id", "metareview_score"]].head(2)

#### Global Row Validation
A row is valid if and only if every condition below is true.

In [ ]:
valid_row = (
    valid_submission_id
    & valid_reviewer_id
    & valid_track
    & valid_assignment_timestamp
    & valid_review_timestamp
    & valid_soundness
    & valid_excitement
    & valid_confidence
    & valid_overall_score
    & valid_metareview_score
    & valid_recommendation
)

In [ ]:
valid_row.value_counts()

In [ ]:
valid_count = valid_row.sum()

valid_count

In [ ]:
malformed_count = (~valid_row).sum()

malformed_count

In [ ]:
valid_count + malformed_count == len(df)

##### Inspect and assert malformed rows

In [ ]:
validation = pd.DataFrame(index=df.index)

validation["invalid_submission_id"] = ~valid_submission_id
validation["invalid_reviewer_id"] = ~valid_reviewer_id
validation["invalid_track"] = ~valid_track
validation["invalid_assignment_timestamp"] = ~valid_assignment_timestamp
validation["invalid_review_timestamp"] = ~valid_review_timestamp
validation["invalid_soundness"] = ~valid_soundness
validation["invalid_excitement"] = ~valid_excitement
validation["invalid_confidence"] = ~valid_confidence
validation["invalid_overall_score"] = ~valid_overall_score
validation["invalid_metareview_score"] = ~valid_metareview_score
validation["invalid_recommendation"] = ~valid_recommendation

In [ ]:
validation.sum()

In [ ]:
validation["is_malformed"] = validation.any(axis=1)

In [ ]:
validation["is_malformed"].sum()

In [ ]:
(validation["is_malformed"] == (~valid_row)).all()

#### Extract valid rows

In [ ]:
clean_df = df.loc[valid_row].copy()

In [ ]:
clean_df.shape

In [ ]:
clean_df.isnull().sum()

In [ ]:
print("Total rows:", len(df))
print("Malformed rows:", validation["is_malformed"].sum())
print("Valid rows:", len(clean_df))